[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/18_embedding_solution.ipynb)

# 🟢 Solution: Embedding Layer（参考版）

## 📋 题目描述

**难度：** Easy

实现嵌入查找层（nn.Module）。

嵌入层通过可学习的权重矩阵将整数索引映射为稠密向量，是 NLP 模型的第一层。

**签名:** `MyEmbedding(num_embeddings, embedding_dim)`（nn.Module）

**前向传播:** `forward(indices) -> Tensor`
- `indices` — 任意形状的整数张量

**返回:** 嵌入向量，末尾增加 embedding_dim 维度

**约束:**
- 权重存储为 `nn.Parameter`，形状 (num_embeddings, embedding_dim)
- 前向传播即 `weight[indices]`

**提示：** `self.weight = nn.Parameter(...)` 形状 `(num_embeddings, embedding_dim)`
前向：`return self.weight[indices]`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        # ---- Step 1: 创建可学习的嵌入矩阵 ----
        # shape: (num_embeddings, embedding_dim)
        # num_embeddings = 词表大小（比如 50000 个 token）
        # embedding_dim = 每个 token 的向量维度（比如 768）
        # nn.Parameter 会自动注册为模型参数，参与梯度更新
        # 用 randn 初始化（标准正态分布），实际中可能用 Xavier 等
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))

    def forward(self, indices):
        # ---- Step 2: 查表操作 ----
        # PyTorch 的高级索引：weight[indices] 直接查表
        # indices 可以是任意 shape (*, )，结果 shape 为 (*, embedding_dim)
        # 例如 indices = [3, 7, 1]，shape (3,)
        #   → weight[indices] 取第 3、7、1 行，shape (3, embedding_dim)
        # 例如 indices shape (2, 4)
        #   → weight[indices] shape (2, 4, embedding_dim)
        # 这等价于 one-hot 编码再矩阵乘法，但高效得多：
        #   one_hot(indices) @ weight  ← 内存开销巨大
        #   weight[indices]            ← 直接取行，零拷贝思路
        return self.weight[indices]


In [ ]:
emb = MyEmbedding(num_embeddings=100, embedding_dim=16)
idx = torch.tensor([1, 5, 99, 42])
print('Output:', emb(idx).shape)
print('Same row:', torch.equal(emb(torch.tensor([5])), emb(torch.tensor([5]))))


In [ ]:
from torch_judge import check
check('embedding')


## 📝 核心思路总结

1. **本质是查表**：Embedding 层就是一个可学习的查找表 (V, D)
2. **高级索引**：`weight[indices]` 等价于 `one_hot @ weight`，但高效得多
3. **稀疏梯度**：反向传播时只有被查到的行收到梯度
4. **vs Linear**：Embedding 是索引查表，Linear 是矩阵乘法
